# DeepSeek-R1 Walkthrough

Goal: reproduce the smallest tensor version of rule rewards, group-relative advantages, and GRPO-style clipped loss.


## Paper-to-code map

Official artifact:

- `learning/reasoning-r1/official/repos/DeepSeek-R1/README.md`
- `learning/reasoning-r1/official/repos/DeepSeek-R1/DeepSeek_R1.pdf`
- `learning/reasoning-r1/official/repos/DeepSeek-R1/figures/benchmark.jpg`

Runnable local source:

- `learning/reasoning-r1/src/grpo_minimal.py`
- `learning/reasoning-r1/src/r1_zero_track_a.py`
- `learning/reasoning-r1/src/r1_zero_track_b.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import torch
import torch.nn.functional as F
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)


## Step 1: group-relative advantage

For each prompt, sample `k` responses and normalize rewards inside the group.


In [ ]:
def compute_group_advantage(rewards, k):
    grouped = rewards.reshape(-1, k)
    return ((grouped - grouped.mean(dim=1, keepdim=True)) / (grouped.std(dim=1, keepdim=True) + 1e-8)).reshape(-1)
rewards = torch.tensor([1.0, 0.0, 0.5, -0.5, 2.0, 2.0, 1.0, 0.0])
A = compute_group_advantage(rewards, k=4)
print("advantages", A)
print("group means", A.reshape(-1, 4).mean(dim=1))


## Step 2: GRPO-style clipped loss


In [ ]:
def grpo_loss(log_probs_new, log_probs_old, log_probs_ref, advantages, response_mask, eps=0.2, beta=0.01):
    ratio = (log_probs_new - log_probs_old).exp()
    adv = advantages.unsqueeze(1)
    clipped = torch.min(ratio * adv, ratio.clamp(1 - eps, 1 + eps) * adv)
    policy_loss = -(clipped * response_mask).sum() / response_mask.sum().clamp(min=1)
    log_r = log_probs_ref - log_probs_new
    kl_loss = ((log_r.exp() - log_r - 1) * response_mask).sum() / response_mask.sum().clamp(min=1)
    return policy_loss + beta * kl_loss, policy_loss, kl_loss
torch.manual_seed(1)
B, k, T = 2, 4, 5
log_new = torch.randn(B * k, T) * 0.1 - 1.0
log_old = log_new + torch.randn(B * k, T) * 0.02
log_ref = log_old.clone()
loss, policy_loss, kl_loss = grpo_loss(log_new, log_old, log_ref, A, torch.ones(B * k, T))
print("loss", round(float(loss), 4), "policy", round(float(policy_loss), 4), "kl", round(float(kl_loss), 4))


## Step 3: rule reward toy


In [ ]:
def toy_rule_reward(response, gold):
    has_format = "<think>" in response and "</think>" in response and "<answer>" in response and "</answer>" in response
    answer_text = response.split("<answer>")[-1].split("</answer>")[0].strip() if "<answer>" in response else ""
    accuracy = float(answer_text == gold)
    format_score = float(has_format)
    return {"format": format_score, "accuracy": accuracy, "total": 0.2 * format_score + 0.8 * accuracy}
for text in ["<think>2+3=5</think><answer>5</answer>", "2+3 is 5", "<think>bad</think><answer>4</answer>"]:
    print(toy_rule_reward(text, "5"))


## Step 4: compare with local implementation and official artifact


In [ ]:
gppo_path = REPO / "learning" / "reasoning-r1" / "src" / "grpo_minimal.py"
spec = importlib.util.spec_from_file_location("grpo_minimal", gppo_path)
grpo_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(grpo_module)
assert torch.allclose(grpo_module.compute_group_advantage(rewards, 4), A)
official_root = REPO / "learning" / "reasoning-r1" / "official" / "repos" / "DeepSeek-R1"
files = [official_root / "README.md", official_root / "DeepSeek_R1.pdf", official_root / "figures" / "benchmark.jpg"]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Closed-book checks

1. Hand-compute group advantages for four rewards from one prompt.
2. Explain why rule rewards work best for verifiable tasks.
3. Explain why the official repository is an artifact rather than a full local training reproduction.
